# Estudo de Ablação: Variações do LS-SVM em Prognose Florestal

Comparamos **3 variantes arquiteturais** do *Least Squares Support Vector Machine* (LS-SVM) sobre exatamente o mesmo `PartitionPlan` para predição de volume de madeira:
1. **LS-SVM Clássico (Kernel RBF)**
2. **LS-SVM com Kernel Quântico**
3. **LS-SVM com Resolvedor Linear Quântico**

In [ ]:
# Instalação automática apenas quando executado no Google Colab
import subprocess, sys
if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/aleerangel/experimental-ml-framework.git",
    ])


In [ ]:
from pathlib import Path
import tempfile
import numpy as np

from experimental_ml.core.enums import MetricDirection, TaskType
from experimental_ml.data import (
    DatasetSchemaSpec,
    create_preparation_plan,
    load_dataset,
    profile_dataset,
    sanitize_dataset,
)
from experimental_ml.features import FeatureScenario, ScenarioMode
from experimental_ml.models import (
    ModelCapabilities,
    ModelFamily,
    ModelSpec,
    validate_model_spec,
)
from experimental_ml.orchestration import load_cell_artifacts, run_experiment_matrix
from experimental_ml.statistics import (
    MultiObjectiveSpec,
    ObjectiveSource,
    ObjectiveSpec,
    compute_pareto_frontier,
    consolidate_descriptive,
    paired_nadeau_bengio_test,
)
from experimental_ml.validation import (
    EvaluationRegime,
    PartitionConfig,
    PartitionStrategy,
    build_partition_plan,
)

## 1. Base de Prognose Florestal e Implementação das Variantes LS-SVM

In [ ]:
class LSSVMVariante:
    def __init__(self, kernel_mode: str = 'rbf_classico', gamma: float = 0.01, reg: float = 10.0) -> None:
        self.kernel_mode = kernel_mode
        self.gamma = gamma
        self.reg = reg
        self.X_train_ = None
        self.alpha_ = None
        self.b_ = 0.0

    def _gram(self, A: np.ndarray, B: np.ndarray) -> np.ndarray:
        if self.kernel_mode == 'kernel_quantico':
            An = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
            Bn = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-12)
            return (np.cos(An @ Bn.T)) ** 2
        sq = np.sum(A ** 2, axis=1, keepdims=True) + np.sum(B ** 2, axis=1) - 2.0 * (A @ B.T)
        return np.exp(-self.gamma * np.maximum(sq, 0.0))

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'LSSVMVariante':
        n = X.shape[0]
        self.X_train_ = X.copy()
        K = self._gram(X, X) + (1.0 / self.reg) * np.eye(n)
        ones = np.ones((n, 1))
        A_mat = np.block([[np.zeros((1, 1)), ones.T], [ones, K]])
        rhs = np.concatenate([[0.0], y])
        sol = np.linalg.solve(A_mat + 1e-9 * np.eye(n + 1), rhs)
        self.b_ = float(sol[0])
        self.alpha_ = sol[1:]
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self._gram(X, self.X_train_) @ self.alpha_ + self.b_

rng = np.random.default_rng(7)
dap = rng.uniform(10.0, 35.0, size=32)
altura = 8.0 + 0.6 * dap + rng.normal(0.0, 1.0, size=32)
idade = rng.integers(3, 12, size=32).astype(float)
volume = 0.00005 * (dap ** 2) * altura + rng.normal(0.0, 0.02, size=32)

schema = DatasetSchemaSpec(target_col='volume', task_type=TaskType.REGRESSION)
bundle = load_dataset({'dap': dap, 'altura': altura, 'idade': idade, 'volume': volume}, schema)
profile = profile_dataset(bundle)
view = sanitize_dataset(bundle, profile, create_preparation_plan(profile, rules=[]))

## 2. Execução Matricial Controlada e Análise Pareto Multiobjetivo

In [ ]:
specs = [
    validate_model_spec(ModelSpec(
        model_id='lssvm_rbf',
        display_name='LS-SVM Clássico (RBF)',
        family=ModelFamily.CLASSICAL_ML,
        estimator_factory=lambda seed=None, **kw: LSSVMVariante(**kw),
        capabilities=ModelCapabilities(requires_fit=True),
        parameters={'kernel_mode': 'rbf_classico', 'gamma': 0.01, 'reg': 10.0},
        description='Kernel RBF clássico',
        scientific_justification='Baseline clássico',
    )),
    validate_model_spec(ModelSpec(
        model_id='lssvm_qkernel',
        display_name='LS-SVM (Kernel Quântico)',
        family=ModelFamily.QUANTUM_KERNEL,
        estimator_factory=lambda seed=None, **kw: LSSVMVariante(**kw),
        capabilities=ModelCapabilities(requires_fit=True, supports_precomputed_kernel=True),
        parameters={'kernel_mode': 'kernel_quantico', 'reg': 10.0},
        description='Kernel quântico de fidelidade',
        scientific_justification='Avaliação de mapa de características quântico',
    )),
    validate_model_spec(ModelSpec(
        model_id='lssvm_qsolver',
        display_name='LS-SVM (Solver Quântico)',
        family=ModelFamily.QUANTUM_SOLVER,
        estimator_factory=lambda seed=None, **kw: LSSVMVariante(**kw),
        capabilities=ModelCapabilities(requires_fit=True),
        parameters={'kernel_mode': 'solver_quantico', 'gamma': 0.01, 'reg': 10.0},
        description='Solver linear quântico',
        scientific_justification='Ablação do solver linear KKT',
    )),
]

cenario = FeatureScenario(
    scenario_id='dendrometria',
    name='Variáveis Dendrométricas',
    description='DAP, altura e idade',
    scientific_justification='Preditores padrão de volume florestal',
    mode=ScenarioMode.FIXED,
    selected_features=('dap', 'altura', 'idade'),
)

part_plan = build_partition_plan(
    view,
    PartitionConfig(strategy=PartitionStrategy.KFOLD, regime=EvaluationRegime.CV_ONLY, n_splits=4, master_seed=42),
)
tmp_dir = Path(tempfile.mkdtemp(prefix='expml_nb02_'))
state = run_experiment_matrix(
    model_specs=specs,
    scenarios=[cenario],
    partition_plan=part_plan,
    sanitized_view=view,
    run_dir=tmp_dir / 'runs',
    metrics=('rmse', 'rmse_rel_pct', 'mbe'),
    master_seed=42,
    overwrite=True,
)
cells = [load_cell_artifacts(cid, tmp_dir / 'runs')[0] for cid in state.cell_statuses]
cons = consolidate_descriptive(cells, total_folds_per_config=len(part_plan.folds))
pareto = compute_pareto_frontier(
    cons,
    MultiObjectiveSpec(objectives=(
        ObjectiveSpec(name='rmse', source=ObjectiveSource.AGGREGATED_METRIC, direction=MetricDirection.MINIMIZE, field_name='rmse'),
        ObjectiveSpec(name='latencia', source=ObjectiveSource.TIMING_MEAN, direction=MetricDirection.MINIMIZE, field_name='predict_mean_s'),
    )),
)
print('Fronteira de Pareto:', [pt.config_id for pt in pareto.pareto_frontier])
assert len(pareto.pareto_frontier) >= 1